# StockSense Round 2 — Feature Engineering & Machine Learning
**Person 2: ML Engineer**

This notebook documents the complete Round 2 ML pipeline:
1. Dataset inspection
2. Feature engineering
3. Target creation
4. Leakage audit
5. Temporal train/val/test split
6. Demand baseline + model comparison
7. Stock-out classification
8. Feature importance
9. Error analysis
10. Final conclusions


## 1. Imports & Setup

In [ ]:
import os, sys, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Add src to path so we can import the project modules
sys.path.insert(0, os.path.join('..', 'src'))
print('Setup complete.')


## 2. Load & Inspect Master Dataset

In [ ]:
master = pd.read_csv('../data/processed/master_daily.csv')
master['date'] = pd.to_datetime(master['date'])
print('Shape:', master.shape)
print('Columns:', list(master.columns))


In [ ]:
print('Date range:', master['date'].min().date(), 'to', master['date'].max().date())
print('Stores:', sorted(master['store_id'].unique()))
print('Products:', sorted(master['product_id'].unique()))
print('Missing values:', master.isnull().sum().sum())
print('Duplicate Date-Store-Product keys:', 
      master.duplicated(['date','store_id','product_id']).sum())


In [ ]:
print('=== Numerical summary ===')
master.describe().round(2)


In [ ]:
print('=== Categorical columns ===')
for col in ['category','sub_category','brand','store_type','region','city']:
    print(f'  {col}: {sorted(master[col].unique())}')


In [ ]:
print('=== Inventory columns ===')
master[['opening','received','sold','closing','reorder_lvl','lead_days']].describe().round(2)


## 3. Feature Engineering
Run the feature engineering module to build all features and targets.

In [ ]:
from feature_engineering import build_features
df = build_features(save=True)
print('Feature matrix shape:', df.shape)
print('New columns added:', [c for c in df.columns if c not in master.columns])


### 3.1 Time Features

In [ ]:
time_cols = ['day_of_week','weekend_flag','month','week_no','festival_flag','holiday_or_festival']
print(df[time_cols].describe().round(2))


### 3.2 Lag & Rolling Features
All computed **within each (store_id, product_id) group**, sorted by date.
No future information is used.

In [ ]:
lag_cols = ['lag_1','lag_7','lag_14','rolling_mean_7','rolling_mean_14','rolling_std_7']
print(df[lag_cols].describe().round(2))

# Example: one store-product pair
sample = df[(df['store_id']=='S01') & (df['product_id']=='P101')].head(20)
sample[['date','units_sold','lag_1','lag_7','rolling_mean_7']]


## 4. Target Creation
### 4.1 Demand Target: `next_7_day_demand`
**Definition**: Sum of `units_sold` for days t+1 through t+7 per (store_id, product_id).
Rows with incomplete 7-day future window → NaN → excluded from training.

In [ ]:
print('next_7_day_demand stats:')
print(df['next_7_day_demand'].describe().round(2))
print('NaN rows (excluded):', df['next_7_day_demand'].isna().sum())
print('Usable rows:', df['next_7_day_demand'].notna().sum())


In [ ]:
# Distribution of next_7_day_demand
fig, ax = plt.subplots(figsize=(10, 4))
df['next_7_day_demand'].dropna().hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribution of next_7_day_demand')
ax.set_xlabel('Units (7-day total)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


### 4.2 Stock-out Target: `stockout_flag`
**Definition**: `1` if `closing < reorder_lvl`, else `0`.
**Rationale**: Below the reorder threshold → imminent stock-out risk.

In [ ]:
print('stockout_flag distribution:')
vc = df['stockout_flag'].value_counts()
print(vc)
print(f'Positive rate: {vc[1]/len(df)*100:.1f}%')


In [ ]:
# Stock-out by store and category
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df.groupby('store_id')['stockout_flag'].mean().plot(kind='bar', ax=axes[0], 
    color='#e74c3c', edgecolor='white')
axes[0].set_title('Stock-out Rate by Store')
axes[0].set_xlabel('Store')
axes[0].set_ylabel('Rate')
axes[0].tick_params(rotation=0)

df.groupby('category')['stockout_flag'].mean().sort_values().plot(kind='barh', ax=axes[1],
    color='#e67e22', edgecolor='white')
axes[1].set_title('Stock-out Rate by Category')
axes[1].set_xlabel('Rate')

plt.tight_layout()
plt.show()


## 5. Data Leakage Audit
Full audit in `reports/ml_leakage_check.md`.

**Key guarantees**:
- Lag features use `.shift(n)` with n≥1 within each (store_id, product_id) group.
- Rolling features use `.shift(1).rolling(n)` — current day excluded.
- `next_7_day_demand` is only the target, never a predictor.
- Inventory features `reorder_gap`, `days_of_inventory`, `stock_coverage_ratio`
  are **excluded from the stockout model** because they algebraically encode the target.


In [ ]:
# Print leakage report summary
with open('../reports/ml_leakage_check.md', 'r') as f:
    content = f.read()
# Show first 2000 chars
print(content[:2000])


## 6. Time-Aware Train / Validation / Test Split
**No random shuffling.** Split by calendar date only.

| Split | Dates | Rows (demand) |
|---|---|---|
| Train | 2026-05-01 to 2026-07-25 | ~4,472 |
| Validation | 2026-07-26 to 2026-08-12 | ~936 |
| Test | 2026-08-13 to 2026-08-24 | ~640 |


In [ ]:
TRAIN_END = '2026-07-25'
VAL_END   = '2026-08-12'

df_model = df.dropna(subset=['next_7_day_demand']).copy()
train = df_model[df_model['date'] <= TRAIN_END]
val   = df_model[(df_model['date'] > TRAIN_END) & (df_model['date'] <= VAL_END)]
test  = df_model[df_model['date'] > VAL_END]

print(f'Train: {len(train)} rows  ({train["date"].min().date()} to {train["date"].max().date()})')
print(f'Val:   {len(val)} rows  ({val["date"].min().date()} to {val["date"].max().date()})')
print(f'Test:  {len(test)} rows  ({test["date"].min().date()} to {test["date"].max().date()})')


## 7. Demand Baseline
**Strategy**: `predicted = rolling_mean_7 × 7`
The naive forecast is the previous 7-day mean scaled to the 7-day window.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def mape(y_true, y_pred):
    mask = y_true > 1e-6
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

y_true_val = val['next_7_day_demand'].values
y_pred_baseline_val = (val['rolling_mean_7'].fillna(0).values * 7)

print('=== Baseline (Validation) ===')
print(f'MAE:  {mean_absolute_error(y_true_val, y_pred_baseline_val):.3f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_true_val, y_pred_baseline_val)):.3f}')
print(f'MAPE: {mape(y_true_val, y_pred_baseline_val):.2f}%')
print(f'R2:   {r2_score(y_true_val, y_pred_baseline_val):.4f}')


## 8. Demand Model Comparison
Models: Linear Regression, Random Forest, XGBoost
Selection metric: **lowest Validation MAE**

In [ ]:
comp = pd.read_csv('../reports/demand_model_comparison.csv')
comp


In [ ]:
# Bar chart: Validation MAE comparison
val_rows = comp[comp['split']=='Validation'].copy()
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#95a5a6' if 'Baseline' in m else '#3498db' for m in val_rows['model']]
ax.bar(val_rows['model'], val_rows['MAE'], color=colors, edgecolor='white')
ax.set_title('Demand Model Comparison — Validation MAE')
ax.set_ylabel('MAE (units)')
ax.set_xlabel('Model')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()


## 9. Selected Demand Model: Random Forest
Selected because it has the **lowest Validation MAE** (26.03 vs 26.13 for XGBoost).
Both RF and XGBoost are essentially tied; RF is chosen by the MAE criterion.

In [ ]:
# Load the saved model
with open('../models/demand_model.pkl', 'rb') as f:
    demand_model = pickle.load(f)
with open('../models/demand_scaler.pkl', 'rb') as f:
    demand_scaler = pickle.load(f)

print('Loaded model:', type(demand_model).__name__)


## 10. Demand Test Set Evaluation

In [ ]:
demand_preds = pd.read_csv('../data/processed/demand_predictions.csv')
demand_preds['date'] = pd.to_datetime(demand_preds['date'])

y_true_te = demand_preds['actual_next_7_day_demand'].values
y_pred_te = demand_preds['predicted_next_7_day_demand'].values

print('=== Random Forest (Test) ===')
print(f'MAE:  {mean_absolute_error(y_true_te, y_pred_te):.3f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_true_te, y_pred_te)):.3f}')
print(f'MAPE: {mape(y_true_te, y_pred_te):.2f}%')
print(f'R2:   {r2_score(y_true_te, y_pred_te):.4f}')


In [ ]:
# Actual vs Predicted scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_true_te, y_pred_te, alpha=0.3, s=15, color='#2196F3')
lims = [min(y_true_te.min(), y_pred_te.min()), max(y_true_te.max(), y_pred_te.max())]
axes[0].plot(lims, lims, 'r--', lw=1, label='Perfect')
axes[0].set_xlabel('Actual next_7_day_demand')
axes[0].set_ylabel('Predicted')
axes[0].set_title('Actual vs Predicted (Test Set)')
axes[0].legend()

errors = y_pred_te - y_true_te
axes[1].hist(errors, bins=40, color='#9C27B0', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Prediction Error (predicted - actual)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution (Test Set)')

plt.tight_layout()
plt.show()


## 11. Stock-out Target Analysis

In [ ]:
print('stockout_flag = 1 when closing < reorder_lvl')
print()
print('Overall distribution:')
print(df['stockout_flag'].value_counts())
print(f'Positive rate: {df["stockout_flag"].mean()*100:.1f}%')


## 12. Class Imbalance Analysis

In [ ]:
# Class imbalance by split
for name, split in [('Train', df[df['date'] <= TRAIN_END]),
                     ('Val',   df[(df['date'] > TRAIN_END) & (df['date'] <= VAL_END)]),
                     ('Test',  df[df['date'] > VAL_END])]:
    pos = split['stockout_flag'].sum()
    pct = pos/len(split)*100
    print(f'{name}: {pos} positives out of {len(split)} ({pct:.1f}%)')

print()
print('Class imbalance handled by: class_weight=balanced (sklearn) + scale_pos_weight (XGBoost)')


## 13. Stock-out Model Comparison
Models: Logistic Regression, Decision Tree, Random Forest, XGBoost
Selection metric: **highest Validation F1** (with Recall as tiebreaker)

In [ ]:
scomp = pd.read_csv('../reports/stockout_model_comparison.csv')
scomp


In [ ]:
# Validation metrics bar chart
val_sc = scomp[scomp['split']=='Validation'].copy()
metrics = ['Accuracy','Precision','Recall','F1','ROC_AUC']
x = np.arange(len(val_sc))
width = 0.15
fig, ax = plt.subplots(figsize=(13, 5))
colors = ['#2196F3','#4CAF50','#FF5722','#9C27B0','#FF9800']
for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax.bar(x + i*width, val_sc[metric], width, label=metric, color=color, alpha=0.8)
ax.set_xticks(x + width*2)
ax.set_xticklabels(val_sc['model'], rotation=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Stock-out Model Comparison — Validation Set')
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 14. Selected Classifier: XGBoost
- Highest Validation F1 = 0.8106
- Recall = 0.8841 (capturing most stock-out cases)
- AUC = 0.9789 (excellent discrimination)

In [ ]:
with open('../models/stockout_model.pkl', 'rb') as f:
    stockout_model = pickle.load(f)
print('Loaded classifier:', type(stockout_model).__name__)


## 15. Stockout Test Evaluation

In [ ]:
spred = pd.read_csv('../data/processed/stockout_predictions.csv')

y_true_s = spred['actual_stockout_flag'].values
y_pred_s = spred['stockout_prediction'].values
y_prob_s = spred['stockout_probability'].values

print('=== XGBoost Classifier (Test) ===')
print(f'Accuracy:  {accuracy_score(y_true_s, y_pred_s):.4f}')
print(f'Precision: {precision_score(y_true_s, y_pred_s):.4f}')
print(f'Recall:    {recall_score(y_true_s, y_pred_s):.4f}')
print(f'F1:        {f1_score(y_true_s, y_pred_s):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_true_s, y_prob_s):.4f}')


## 16. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true_s, y_pred_s)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['No Risk (0)', 'Stock-out Risk (1)'])
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Confusion Matrix — XGBoost (Test Set)')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')
print(f'Missed stock-outs (FN): {fn}')
print(f'False alarms (FP): {fp}')


## 17. Feature Importance

In [ ]:
# Demand model feature importance
demand_fi = pd.read_csv('../reports/demand_feature_importance.csv')
print('Top 15 features — Demand (RandomForest):')
print(demand_fi.head(15).to_string(index=False))


In [ ]:
# Stockout model feature importance
stockout_fi = pd.read_csv('../reports/stockout_feature_importance.csv')
print('Top 15 features — Stockout (XGBoost):')
print(stockout_fi.head(15).to_string(index=False))


In [ ]:
# Side-by-side importance charts
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top_d = demand_fi.head(15)
axes[0].barh(top_d['feature'][::-1], top_d['importance'][::-1], color='#2196F3')
axes[0].set_title('Top 15 Features — Demand Model (RandomForest)')
axes[0].set_xlabel('Importance')
axes[0].grid(axis='x', alpha=0.3)

top_s = stockout_fi.head(15)
axes[1].barh(top_s['feature'][::-1], top_s['importance'][::-1], color='#FF5722')
axes[1].set_title('Top 15 Features — Stockout Model (XGBoost)')
axes[1].set_xlabel('Importance')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


## 18. Error Analysis

In [ ]:
# Demand errors
demand_preds['error'] = demand_preds['actual_next_7_day_demand'] - demand_preds['predicted_next_7_day_demand']
demand_preds['abs_error'] = demand_preds['error'].abs()

print('=== High-error products ===')
print(demand_preds.groupby('product_id')['abs_error'].mean().sort_values(ascending=False).head(5))

print()
print('=== High-error stores ===')
print(demand_preds.groupby('store_id')['abs_error'].mean().sort_values(ascending=False))


In [ ]:
# Error over time
daily_err = demand_preds.groupby('date')['abs_error'].mean()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily_err.index, daily_err.values, color='#9C27B0', linewidth=1.5)
ax.fill_between(daily_err.index, daily_err.values, alpha=0.2, color='#9C27B0')
ax.set_title('Mean Absolute Error Over Time (Test Period)')
ax.set_xlabel('Date')
ax.set_ylabel('Mean Absolute Error')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Stockout probability distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(spred['stockout_probability'], bins=40, color='#FF5722', edgecolor='white')
axes[0].set_title('Distribution of stockout_probability')
axes[0].set_xlabel('Probability')
axes[0].set_ylabel('Count')

# Predicted probability for actual positives vs negatives
axes[1].hist(spred[spred['actual_stockout_flag']==0]['stockout_probability'],
             bins=30, alpha=0.7, label='No Risk', color='#4CAF50')
axes[1].hist(spred[spred['actual_stockout_flag']==1]['stockout_probability'],
             bins=30, alpha=0.7, label='Stock-out Risk', color='#F44336')
axes[1].set_title('Stockout Probability by Actual Class')
axes[1].set_xlabel('Predicted Probability')
axes[1].legend()

plt.tight_layout()
plt.show()


## 19. Final ML Conclusions

### Demand Model
| | Validation | Test |
|---|---|---|
| Baseline (rolling mean ×7) | MAE=30.71 | MAE=55.50 |
| **Random Forest (SELECTED)** | **MAE=26.03** | **MAE=51.87** |
| XGBoost | MAE=26.13 | MAE=51.92 |
| Linear Regression | MAE=29.05 | MAE=79.93 |

- **RandomForest** beats the baseline by ~15% on validation MAE.
- The dominant feature is `rolling_mean_14` (88.7% of importance).
- R² = 0.947 on test — model explains 94.7% of demand variance.

### Stockout Model
| | Val F1 | Test F1 | Test AUC |
|---|---|---|---|
| Logistic Regression | 0.596 | 0.514 | 0.862 |
| Decision Tree | 0.603 | 0.520 | 0.815 |
| Random Forest | 0.620 | 0.566 | 0.905 |
| **XGBoost (SELECTED)** | **0.811** | **0.698** | **0.950** |

- **XGBoost** dominates on all metrics.
- Recall = 72.8% on test — captures 73% of true stock-out cases.
- 44 false negatives (missed stock-outs) out of 162 positives.

### Key Limitations
1. Only 123 days of history — no annual seasonality captured.
2. P999 has only 11 days of data (cold-start product) — degraded accuracy.
3. Demand model has positive bias (mean error = +37.6 units) → slight under-prediction.

### Handoff to Person 3
- `models/demand_model.pkl` + `demand_scaler.pkl` → demand forecasting
- `models/stockout_model.pkl` + `stockout_scaler.pkl` → stockout risk scoring
- `data/processed/demand_predictions.csv` → demand forecast output
- `data/processed/stockout_predictions.csv` → stockout risk with probabilities
- Threshold tuning: adjust classification threshold on `stockout_probability` to tune Precision/Recall tradeoff.

**PERSON 3 CAN NOW START.**
